# Final comparison

Loads standardized metrics JSON files from `results/`.

Current expected paths:
- `results/tim/tim_vit_metrics.json`
- `results/hannah_scratch_cnn_metrics.json`
- `results/xenia_resnet_metrics.json`

(For backward compatibility, Tim's legacy path `results/tim_vit_metrics.json` is also checked.)

In [ ]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path.cwd().resolve()
# Allow running with cwd = ieor142b/ or a subfolder
for base in [ROOT, *ROOT.parents]:
    if (base / "results").is_dir() and (base / "MovieGenre.csv").is_file():
        ROOT = base
        break

candidate_paths = {
    "vit": [
        ROOT / "results" / "tim" / "tim_vit_metrics.json",
        ROOT / "results" / "tim_vit_metrics.json",  # legacy path
    ],
    "scratch_cnn": [ROOT / "results" / "hannah_scratch_cnn_metrics.json"],
    "resnet_ft": [ROOT / "results" / "xenia_resnet_metrics.json"],
}

rows = []
for label, plist in candidate_paths.items():
    p = next((x for x in plist if x.is_file()), None)
    if p is None:
        print("Missing (run experiment notebook first):", plist[0])
        continue
    d = json.loads(p.read_text(encoding="utf-8"))
    d["_key"] = label
    rows.append(d)

if not rows:
    raise SystemExit("No metrics files found. Save JSON from each experiment notebook first.")

df = pd.DataFrame(rows)
cols = [
    "model_name",
    "test_f1",
    "test_precision",
    "test_recall",
    "test_exact_match",
    "best_val_f1",
    "num_epochs_run",
    "train_size",
    "val_size",
    "test_size",
    "notes",
]
cols = [c for c in cols if c in df.columns]

df.sort_values("test_f1", ascending=False)[cols]